# 🍷 Wine Recommender System
Porównanie modeli rekomendacji: NCF, Gradient Boosting, Random Forest (PyTorch) oraz RP3β

## 1. Importy i konfiguracja

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Konfiguracja
DATA_PATH   = r'data\XWines_Slim_150K_ratings.csv'
BATCH_SIZE  = 2048
EMBED_DIM   = 50
RANDOM_SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Ładowanie i podział danych

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=['Rating', 'UserID', 'WineID', 'Date'])
df.sort_values('Date', ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Wiersze: {len(df):,} | Unikalni użytkownicy: {df.UserID.nunique():,} | Unikalne wina: {df.WineID.nunique():,}')
df.head()

In [ ]:
# Podział chronologiczny 80/20
split_idx = int(len(df) * 0.8)
train_df_raw = df.iloc[:split_idx].copy()
test_df_raw  = df.iloc[split_idx:].copy()

# Słowniki indeksów (budowane tylko na zbiorze treningowym)
user2idx = {u: i for i, u in enumerate(train_df_raw['UserID'].unique())}
wine2idx = {w: i for i, w in enumerate(train_df_raw['WineID'].unique())}

num_users = len(user2idx)
num_wines = len(wine2idx)
print(f'Użytkownicy w train: {num_users:,} | Wina w train: {num_wines:,}')


def map_and_clean(df_split, user2idx, wine2idx, rating_col='Rating'):
    """Mapuje ID → indeks i usuwa cold-start (NaN po mapowaniu)."""
    df_split = df_split.copy()
    df_split['user_idx'] = df_split['UserID'].map(user2idx)
    df_split['wine_idx'] = df_split['WineID'].map(wine2idx)
    mask = df_split['user_idx'].notna() & df_split['wine_idx'].notna()
    df_split = df_split[mask].copy()
    df_split['user_idx'] = df_split['user_idx'].astype(int)
    df_split['wine_idx'] = df_split['wine_idx'].astype(int)
    return df_split


X_train = map_and_clean(train_df_raw, user2idx, wine2idx)
X_test  = map_and_clean(test_df_raw,  user2idx, wine2idx)

y_train = X_train['Rating']
y_test  = X_test['Rating']

print(f'Train: {len(X_train):,} | Test: {len(X_test):,} '
      f'({len(test_df_raw) - len(X_test):,} próbek odrzuconych – cold start)')

## 3. Konwersja do tensorów PyTorch

In [ ]:
def to_tensors(df_split, y_series):
    """Zwraca (user_tensor, wine_tensor, rating_tensor)."""
    users   = torch.tensor(df_split['user_idx'].values, dtype=torch.long)
    wines   = torch.tensor(df_split['wine_idx'].values, dtype=torch.long)
    ratings = torch.tensor(y_series.values,             dtype=torch.float32)
    return users, wines, ratings


X_user_train, X_wine_train, y_train_t = to_tensors(X_train, y_train)
X_user_test,  X_wine_test,  y_test_t  = to_tensors(X_test,  y_test)

train_loader = DataLoader(
    TensorDataset(X_user_train, X_wine_train, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_user_test, X_wine_test, y_test_t),
    batch_size=BATCH_SIZE
)

print(f'Batche treningowe: {len(train_loader)} | testowe: {len(test_loader)}')

## 4. Funkcje metryk

In [ ]:
# ─── Metryki regresji ───────────────────────────────────────────────────────

def regression_metrics(y_true, y_pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'R2':   r2_score(y_true, y_pred),
    }


# ─── Metryki rankingowe (na poziomie elementów) ─────────────────────────────

def precision_at_k(y_true, y_pred, k=10, threshold=4.0):
    """Ile z top-K rekomendacji to trafione (ocena ≥ threshold)."""
    idx = np.argsort(y_pred)[::-1][:k]
    return np.sum(y_true[idx] >= threshold) / k


def recall_at_k(y_true, y_pred, k=10, threshold=4.0):
    """Jaka część wszystkich dobrych win znalazła się w top-K."""
    idx = np.argsort(y_pred)[::-1][:k]
    total_relevant = np.sum(y_true >= threshold)
    return np.sum(y_true[idx] >= threshold) / total_relevant if total_relevant else 0


def ndcg_at_k(y_true, y_pred, k=10):
    """Normalized Discounted Cumulative Gain – uwzględnia pozycję elementu."""
    def dcg(r):
        r = np.asarray(r, dtype=np.float64)[:k]
        return np.sum(r / np.log2(np.arange(2, r.size + 2))) if r.size else 0.0

    actual_dcg = dcg(y_true[np.argsort(y_pred)[::-1]])
    ideal_dcg  = dcg(np.sort(y_true)[::-1])
    return actual_dcg / ideal_dcg if ideal_dcg else 0


# ─── Metryki rankingowe uśrednione po użytkownikach ─────────────────────────

def ranking_metrics_per_user(X_df, y_true_arr, y_pred_arr, k=10):
    df_eval = X_df[['user_idx']].copy()
    df_eval['y_true'] = y_true_arr
    df_eval['y_pred'] = y_pred_arr

    p_scores, r_scores, n_scores = [], [], []
    for _, user_data in df_eval.groupby('user_idx'):
        if len(user_data) < k:
            continue
        yt = user_data['y_true'].values
        yp = user_data['y_pred'].values
        p_scores.append(precision_at_k(yt, yp, k))
        r_scores.append(recall_at_k(yt, yp, k))
        n_scores.append(ndcg_at_k(yt, yp, k))

    return {
        f'Precision@{k}': np.mean(p_scores),
        f'Recall@{k}':    np.mean(r_scores),
        f'NDCG@{k}':      np.mean(n_scores),
    }


# ─── Catalog Coverage ───────────────────────────────────────────────────────

def catalog_coverage(model, user2idx, wine2idx, device, threshold=3.5, sample_users=100):
    """Jaki procent win model potrafi zarekomendować co najmniej jednemu userowi."""
    model.eval()
    recommended = set()
    sampled = np.random.choice(list(user2idx.values()), min(sample_users, len(user2idx)), replace=False)
    wines_t = torch.tensor(list(wine2idx.values()), dtype=torch.long, device=device)

    with torch.no_grad():
        for u in sampled:
            users_t = torch.full_like(wines_t, u)
            preds = model(users_t, wines_t).squeeze().cpu().numpy()
            recommended.update(np.where(preds >= threshold)[0])

    return len(recommended) / len(wine2idx)


# ─── Kompletna ewaluacja modelu PyTorch ─────────────────────────────────────

def evaluate_pytorch_model(model, test_loader, X_test_df, y_test_arr,
                           user2idx, wine2idx, device, model_name):
    model.eval()
    all_preds, all_true = [], []

    with torch.no_grad():
        for user, wine, rating in test_loader:
            preds = model(user.to(device), wine.to(device)).squeeze().cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(rating.numpy())

    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)

    return {
        'Model': model_name,
        **regression_metrics(all_true, all_preds),
        **ranking_metrics_per_user(X_test_df, all_true, all_preds, k=5),
        **ranking_metrics_per_user(X_test_df, all_true, all_preds, k=10),
        'Coverage': catalog_coverage(model, user2idx, wine2idx, device),
    }, all_preds


print('✓ Funkcje metryk załadowane.')

## 5. Generyczna pętla treningowa

In [ ]:
def train_model(model, train_loader, test_loader, *,
                epochs=20, lr=1e-3, weight_decay=1e-5,
                patience=3, save_path=None,
                clip_grad=None, use_scheduler=False):
    """
    Generyczna pętla train/eval z early stopping.
    Zwraca model z najlepszymi wagami.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    ) if use_scheduler else None

    best_loss = float('inf')
    counter   = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        # --- TRAIN ---
        model.train()
        train_loss = 0.0
        for user, wine, rating in train_loader:
            user, wine, rating = user.to(device), wine.to(device), rating.to(device)
            optimizer.zero_grad()
            preds = model(user, wine).squeeze()
            loss  = criterion(preds, rating)
            loss.backward()
            if clip_grad:
                nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # --- EVAL ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for user, wine, rating in test_loader:
                user, wine, rating = user.to(device), wine.to(device), rating.to(device)
                preds = model(user, wine).squeeze()
                test_loss += criterion(preds, rating).item()
        test_loss /= len(test_loader)

        if scheduler:
            scheduler.step(test_loss)

        tag = ''
        if test_loss < best_loss:
            best_loss  = test_loss
            counter    = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if save_path:
                torch.save(best_state, save_path)
            tag = '  ✓ best'
        else:
            counter += 1

        print(f'Epoch {epoch:3d}/{epochs}  train={train_loss:.4f}  test={test_loss:.4f}{tag}')

        if counter >= patience:
            print(f'  Early stopping po epoce {epoch}.')
            break

    if best_state:
        model.load_state_dict(best_state)
    print(f'\n✓ Najlepszy test loss: {best_loss:.4f}  (RMSE ≈ {best_loss**0.5:.4f})')
    return model


print('✓ Pętla treningowa załadowana.')

## 6. Model NCF (Neural Collaborative Filtering)

In [ ]:
class NCF(nn.Module):
    """Neural Collaborative Filtering – MLP na embeddings użytkownik × wino."""

    def __init__(self, num_users, num_wines, embedding_dim=50):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.wine_emb = nn.Embedding(num_wines, embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.wine_emb.weight, std=0.01)

        self.net = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, user, wine):
        x = torch.cat([self.user_emb(user), self.wine_emb(wine)], dim=1)
        return self.net(x)


model_ncf = NCF(num_users, num_wines, embedding_dim=EMBED_DIM)
print(model_ncf)

In [ ]:
model_ncf = train_model(
    model_ncf, train_loader, test_loader,
    epochs=50, lr=3e-3, weight_decay=1e-4,
    patience=5, save_path='best_ncf.pt',
    clip_grad=1.0, use_scheduler=True,
)

In [ ]:
metrics_ncf, preds_ncf = evaluate_pytorch_model(
    model_ncf, test_loader, X_test, y_test.values,
    user2idx, wine2idx, device, 'NCF'
)
print(metrics_ncf)

## 7. Model Gradient Boosting (PyTorch)

In [ ]:
class GradientBoostingPyTorch(nn.Module):
    """
    Aproksymacja gradient boostingu – suma n_trees słabych sieci,
    każda uczona ze współczynnikiem learning_rate.
    """

    def __init__(self, num_users, num_wines, embedding_dim=32, n_trees=50, lr_tree=0.05):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.wine_emb = nn.Embedding(num_wines, embedding_dim)
        self.lr_tree  = lr_tree

        def weak_learner():
            return nn.Sequential(
                nn.Linear(embedding_dim * 2, 64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32),                nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(32, 1),
            )

        self.trees = nn.ModuleList([weak_learner() for _ in range(n_trees)])

    def forward(self, user, wine):
        x = torch.cat([self.user_emb(user), self.wine_emb(wine)], dim=1)
        return sum(self.lr_tree * tree(x) for tree in self.trees)


model_gb = GradientBoostingPyTorch(num_users, num_wines, embedding_dim=32, n_trees=50)
print(model_gb)

In [ ]:
model_gb = train_model(
    model_gb, train_loader, test_loader,
    epochs=20, lr=1e-3, weight_decay=1e-5, patience=3,
)

In [ ]:
metrics_gb, preds_gb = evaluate_pytorch_model(
    model_gb, test_loader, X_test, y_test.values,
    user2idx, wine2idx, device, 'Gradient Boosting'
)
print(metrics_gb)

## 8. Model Random Forest (PyTorch)

In [ ]:
class RandomForestPyTorch(nn.Module):
    """
    Aproksymacja random forest – uśrednienie n_estimators niezależnych sieci
    z agresywnym dropoutem jako mechanizmem różnicowania.
    """

    def __init__(self, num_users, num_wines, embedding_dim=32, n_estimators=30):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.wine_emb = nn.Embedding(num_wines, embedding_dim)

        def tree():
            return nn.Sequential(
                nn.Linear(embedding_dim * 2, 128), nn.ReLU(), nn.Dropout(0.5),
                nn.Linear(128, 64),                nn.ReLU(), nn.Dropout(0.4),
                nn.Linear(64, 32),                 nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(32, 1),
            )

        self.trees = nn.ModuleList([tree() for _ in range(n_estimators)])

    def forward(self, user, wine):
        x = torch.cat([self.user_emb(user), self.wine_emb(wine)], dim=1)
        return torch.stack([t(x) for t in self.trees]).mean(dim=0)


model_rf = RandomForestPyTorch(num_users, num_wines, embedding_dim=32, n_estimators=30)
print(model_rf)

In [ ]:
model_rf = train_model(
    model_rf, train_loader, test_loader,
    epochs=20, lr=1e-3, weight_decay=1e-5,
    patience=3, save_path='best_rf.pt',
)

In [ ]:
metrics_rf, preds_rf = evaluate_pytorch_model(
    model_rf, test_loader, X_test, y_test.values,
    user2idx, wine2idx, device, 'Random Forest'
)
print(metrics_rf)

## 9. Model RP3β (klasyczny collaborative filtering)

In [ ]:
class RP3Beta:
    """Random Walk z restartem i tłumieniem degree bias (β)."""

    def __init__(self, beta=0.6, top_k=100):
        self.beta  = beta
        self.top_k = top_k

    def fit(self, df, user_col='UserID', item_col='WineID', rating_col='Rating'):
        self.users    = df[user_col].unique()
        self.items    = df[item_col].unique()
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.item2idx = {it: j for j, it in enumerate(self.items)}

        rows = df[user_col].map(self.user2idx).values
        cols = df[item_col].map(self.item2idx).values
        self.R = csr_matrix(
            (df[rating_col].values, (rows, cols)),
            shape=(len(self.users), len(self.items))
        )
        self.item_degree = np.array(self.R.sum(axis=0)).flatten() ** self.beta
        return self

    def recommend(self, user_id, top_n=None):
        top_n  = top_n or self.top_k
        u_idx  = self.user2idx.get(user_id)
        if u_idx is None:
            return []

        user_row = self.R[u_idx].toarray().flatten()
        scores   = (self.R.T @ self.R) @ user_row / self.item_degree
        scores[user_row > 0] = -np.inf   # ukryj już ocenione

        top_idx = np.argsort(scores)[::-1][:top_n]
        return list(zip(self.items[top_idx], scores[top_idx]))


# ─── Metryki RP3β (listy, nie tablice NumPy) ────────────────────────────────

def _precision_list(rec, rel, k):
    return len(set(rec[:k]) & set(rel)) / k

def _recall_list(rec, rel, k):
    return len(set(rec[:k]) & set(rel)) / len(rel) if rel else 0

def _hit_rate_list(rec, rel, k):
    return int(bool(set(rec[:k]) & set(rel)))


def evaluate_rp3beta(model, test_df, k=10):
    p_scores, r_scores, h_scores = [], [], []

    for user, grp in test_df.groupby('UserID'):
        if user not in model.user2idx:
            continue
        relevant = grp['WineID'].tolist()
        rec_items = [item for item, _ in model.recommend(user, top_n=k)]

        p_scores.append(_precision_list(rec_items, relevant, k))
        r_scores.append(_recall_list(rec_items,    relevant, k))
        h_scores.append(_hit_rate_list(rec_items,  relevant, k))

    return {
        f'Precision@{k}': np.mean(p_scores),
        f'Recall@{k}':    np.mean(r_scores),
        f'HitRate@{k}':   np.mean(h_scores),
    }


print('✓ RP3β załadowany.')

In [ ]:
# Podział per-user (losowy) – osobny dla RP3β
def user_split(df, test_size=0.2, random_state=RANDOM_SEED):
    train_parts, test_parts = [], []
    for _, grp in df.groupby('UserID'):
        if len(grp) < 2:
            continue
        tr, te = train_test_split(grp, test_size=test_size, random_state=random_state)
        train_parts.append(tr)
        test_parts.append(te)
    return pd.concat(train_parts), pd.concat(test_parts)


rp3_train, rp3_test = user_split(df)

rp3 = RP3Beta(beta=0.6, top_k=10).fit(rp3_train)
metrics_rp3 = evaluate_rp3beta(rp3, rp3_test, k=10)

print('Wyniki RP3β:')
for k, v in metrics_rp3.items():
    print(f'  {k}: {v:.4f}')

## 10. Podsumowanie i wizualizacja wyników

In [ ]:
DISPLAY_METRICS = ['RMSE', 'R2', 'Precision@10']
COLORS          = ['#4C72B0', '#DD8452', '#55A868']

results_df = (
    pd.DataFrame([metrics_ncf, metrics_gb, metrics_rf])
    .set_index('Model')
    [DISPLAY_METRICS]
)

print('PORÓWNANIE MODELI')
print('=' * 55)
print(results_df.to_string(float_format='{:.4f}'.format))
print('=' * 55)

print('\nNajlepszy model według metryki:')
for metric in DISPLAY_METRICS:
    if metric == 'RMSE':
        best = results_df[metric].idxmin()
        val  = results_df[metric].min()
    else:
        best = results_df[metric].idxmax()
        val  = results_df[metric].max()
    print(f'  {metric:14s}: {best:22s} ({val:.4f})')


# ─── Wykres ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(DISPLAY_METRICS), figsize=(5 * len(DISPLAY_METRICS), 5))
fig.suptitle('Porównanie modeli rekomendacji', fontsize=15, fontweight='bold')

for ax, metric, color in zip(axes, DISPLAY_METRICS, COLORS):
    values = results_df[metric].values
    bars = ax.bar(results_df.index, values, color=COLORS, edgecolor='white', linewidth=0.8)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h, f'{h:.4f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
print('\nWykres zapisany → model_comparison.png')
plt.show()

results_df.to_csv('model_metrics.csv')
print('Metryki zapisane → model_metrics.csv')